# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

請查閱 Python 文件或自行測試，完成下表。

| 數學概念 | Python 型別 | 範例 |
|----------|------------|------|
| 自然數 | int | 5 |
| 整數 | int | 10,0,-5 |
| 有理數 | fractions.Fraction |Fraction(3, 4) |
| 實數 | float | 3.14159 |
| 複數 | complex | 3 + 4i |
| 向量 | list,tuple | [1, 2, 3] |
| 矩陣 | list 的嵌套 | [[1, 2], [3, 4]] |
| 集合 | set,frozenset | {1, 2, 3}, frozenset({1, 2}) |
| 命題真假 | bool | True, False |

並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？int,float,complex,list、tuple,set,frozenset,bool
2. 哪些需要額外模組才能較完整地表示？有理數,精確實數,向量與矩陣
3. Python 的型別分類與數學中的分類是否完全一致？不完全一致

---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？否，Python 沒有為每一個數系都提供獨立的內建型別
2. 為什麼自然數與整數都使用 `int` 表示？主要原因來自程式語言設計的簡化性與運算封閉性
3. Python 為什麼提供 `complex` 型別？因其科學計算與工程應用定位
4. Python 為什麼沒有內建的有理數型別？因效能與記憶體開銷
5. 下列三個物件的型別有何不同? 分別為整數,浮點數,複數
```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | 併入整數實作，無獨立結構體；對於小整數（-5 到 256）會使用快取機制避免重複建立物件。 |
| 整數 | `int` | PyLongObject | 底層將大整數拆解為陣列，實現無上限的整數運算 |
| 實數 | `float` | PyFloatObject | IEEE 754 雙精度浮點數。結構體內部直接包裝一個 C 語言的 double 型態變數 |
| 複數 | `complex` | PyComplexObject | double 組合。內部由一個 Py_complex 結構體組成，直接包含兩個 C 語言 double 變數 |
| 集合 | `set` | PySetObject | 底層由 entry 陣列構成，每個 entry 存放 key 與其 hash 值；利用開放尋址法與二次探測解決雜湊衝突，實現平均$O(1)$的尋找與插入 |
| 命題真假 | `bool` | PyLongObject | int 的子型別。底層完全重用 PyLongObject，全域僅存在兩個單例物件 |
| 串列 | `list` | PyListObject | 動態指標陣列。內部維護一個指向 PyObject的連續記憶體陣列；採用預先分配策略，支援平均 $O(1)$ 的尾端 Append/Pop 運算。 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？都是
2. 對應的 CPython 物件結構名稱為何？
自然數 / 整數(PyLongObject) 
實數(PyFloatObject) 
複數(PyComplexObject)
集合(PySetObject)
命題真假(PyLongObject)
串列(PyListObject)
3. 它如何儲存資料？
自然數 / 整數:採用任意精度整數機制
實數:內部直接包裹一個C語言原生的 double 變數，符合IEEE 754雙精度浮點數標準
複數:內部包裝一個 C 結構體 Py_complex，包含兩個 C 語言原生的 double 變數，分別代表實部與虛部
集合:使用雜湊表
命題真假:直接繼承並重用 PyLongObject 的結構
串列:採用動態指標陣列
4. 是否有固定大小限制？
有:實數,複數,命題真假
無:自然數 / 整數,集合,串列
5. 若資料規模持續增加，如何擴充儲存空間？
無法擴充:實數,複數,命題真假
自然數 / 整數:當數值變大超過當前陣列能容納的範圍時，Python 會重新分配一塊更大的記憶體，建立新的 PyLongObject 並複製資料
集合:採用自動重雜湊機制
串列:採用過度分配與動態擴容策略
---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？因為 Python 3 的 int 型別採用了動態長度與任意精度的演算法結構
2. Python 的 `int` 是否具有固定的位元數限制？沒有固定的位元數限制
3. 什麼是 Arbitrary Precision（任意精度）？指計算機系統在處理數值計算時不受到硬體 CPU 暫存器固定長度（如 64 bit）的限制,精確度僅取決於可用記憶體大小
4. CPython 使用什麼物件來實作 `int`？CPython 使用 PyLongObject 結構體來實作 int
5. Python 整數大小真正的限制是什麼？記憶體
---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？因為電腦內部採用二進位來儲存資料，而某些十進位小數無法在二進位中被有限位數精確表示
2. 浮點數是否能精確表示所有小數？不能
3. 什麼是浮點數的精度（precision）？精度是指浮點數能精確紀錄有效數字的位數
4. 什麼是浮點數的範圍（range）？範圍是指浮點數能表示的最大數值與最小數值
5. 精度與範圍有何不同？範圍（Range）決定了「數字能滾到多大或多小」（界線在哪裡,精度（Precision）決定了「數字能刻劃得有多細緻」（細節到哪裡）


---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？AI 模型中的權重絕大多數使用浮點數型別（Floating-point formats）儲存
2. 什麼是 FP64？雙精度浮點數,即 Python 的 float 與 C 語言的 double。擁有極高的精度與極大的動態範圍，但記憶體佔用極大（每個權重佔 8 Bytes），計算速度慢且極耗電
3. 什麼是 FP32？單精度浮點數,長期以來作為深度學習的「標準/全精度（Full Precision）」格式
4. 什麼是 FP16？半精度浮點數,每個權重僅佔 2 Bytes，可直接將記憶體需求折半並顯著提升計算吞吐量
5. 什麼是 BF16？專為 AI 深度學習設計。它保留了與 FP32 完全相同的指數位數（8 位元），犧牲部分尾數精度（7 位元），使得它擁有與 FP32 一樣寬廣的動態範圍，同時將體積縮減至 2 Bytes
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？神經網路需要「微小的連續變化」,數值空間的非線性放大,Python int 物件開銷極度龐大

---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int |動態可變 | 無誤差 | 流程控制、精確計數、索引存取、密碼學大數運算 |
| FP64 | 64 bit | 有誤差 | 高精度科學計算、物理模擬、航太工程、地理資訊系統 |
| FP32 | 32 bit | 有誤差 | 傳統電腦繪圖（3D 渲染）、傳統深度學習訓練、一般科學計算 |
| FP16 | 16 bit | 有誤差 | 傳統 GPU 加速、邊緣裝置 AI 推論、早期混合精度訓練 |
| BF16 | 16 bit | 有誤差 | 現代大型 AI 模型（LLM）的訓練與推論 |

並回答：

1. 哪種表示方式最精確？整數精確度：int 最精確。小數/浮點數精確度：FP64 最精確
2. 哪種表示方式最節省記憶體？FP16 與 BF16 最節省記憶體。
3. 哪種表示方式最常見於 AI 訓練？BF16（最主流）與 FP32
4. 哪種表示方式最常見於 AI 推論（Inference）？BF16 / FP16 為標準未量化部署的首選

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？在程式語言的語法與抽象層面上，整數型別沒有固定數值的上限或下限，也沒有固定字元數（位元數）的限制
2. 它是否代表可以儲存無限大的整數？數學理論上「是」，但實體物理世界上「否」
3. CPython 如何實現這項特性？CPython 透過 動態陣列結構（Dynamic Array Struct） 與 大數運算演算法（Bignum Arithmetic） 來實現

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？因為 int 採用了 軟體層面的動態記憶體管理
2. 為什麼 `float` 卻存在範圍限制？因為 float 採用了 硬體層面的固定維度格式
3. 為什麼 `float` 可能產生誤差？float 產生誤差的原因來自於 進位制轉換問題 與 固定位元數的截斷

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。Python數學正確性重於一切。程式語言不應該因為硬體限制而產出錯誤或溢位的計算結果,AI 模型的 FP16 / BF16在容許一定誤差前提下，追求單位時間與單位功耗下的極致計算吞吐量（FLOPS）與記憶體頻寬利用率
2. 各自的優點與缺點
Python的優點是零計算誤差程式邏輯極度安全但計算效能極差記憶體開銷龐大
AI 模型常用的 FP16 / BF16的優點是極高吞吐量,極省記憶體,低功耗但存在精度捨入誤差,可能發生數值不穩定

3. 為什麼大型語言模型通常選擇較低精度的表示方式？LLM 的運算本質是機率統計與高維空間的向量投影，並非精確的符號邏輯推導。神經網路的權重包含數百億到數千億個參數，單一參數極微小的精度損失（例如 $0.0012345$ 變成 $0.001235$）會被全網域的統計分佈吸收，不會影響最終輸出語意
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？引發嚴重的硬體與經濟災難
---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
- 計算速度
- 數值精度
- GPU 運算效率

等角度進行討論。

int 並非直接儲存在 CPU 的 32/64 位元暫存器裡，而是被包裹為 PyLongObject 結構體，本質上是一個動態陣列,若改用 Python 的任意精度整數或 FP64 等高精度格式，將面臨嚴重的硬體瓶頸

|角度|Python int | FP64 | FP32 | FP16 / BF16 |
|----------|--------|------------|----------|----------|
| 設計哲學 |  正確性優先 |絕對精確 | 傳統精度 | 效能與吞吐量極致化 |
| 大小/空間 | 動態擴充 | 8 Bytes | 4 Bytes | 2 Bytes |
| 運算速度 | 極慢 | 慢 | 中等 | 極快 |
|AI 適用性| 記憶體與算力崩潰 | 效益極低且資源浪費 | 僅用於部分核心訓練 | 現今大語言模型主流 |